# 🧠 Self-Reflection & Critique — Hands-On Tutorial
**Boston Institute of Analytics | BIA Lecture 11**

This notebook provides a complete, hands-on implementation of all three self-reflection techniques from the lecture:

| # | Technique | Purpose |
|---|-----------|--------|
| 1 | **Reflexion Algorithm** | Self-corrective retry loop |
| 2 | **Auto-Verbalization Grading** | Self-assessed confidence scoring |
| 3 | **AutoGen Evaluator Sub-Agent** | Structured peer-review critique |

**LLM Backend:** Gemma 4 E2B — local inference via Ollama (`http://localhost:11434`)

---

## ⚙️ Setup — Install Dependencies & Connect to Local Gemma 4 Server

In [2]:
import sys
# Install openai into the active kernel environment
!{sys.executable} -m pip install -q openai

In [3]:
from openai import OpenAI, APIConnectionError
import json
import textwrap
import time

# ─────────────────────────────────────────────────────────────
# 🚀  Local Gemma 4 via Ollama — no API key needed!
#     Ensure the server is running before executing this cell:
#       brew services start ollama
#       ollama pull gemma4:e2b   (one-time, ~7.2 GB download)
# ─────────────────────────────────────────────────────────────
MODEL = "gemma4:e2b"

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",   # required by the SDK but not validated by Ollama
)

# Helper: call Gemma 4 and return text — retries on transient connection errors
def call_gemma(
    prompt: str,
    system: str = None,
    model_name: str = MODEL,
    max_retries: int = 3,
    retry_delay: float = 5.0
) -> str:
    """OpenAI-compatible call to the local Ollama / Gemma 4 server.
    Retries up to max_retries times on APIConnectionError with linear backoff.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                temperature=1.0,
                top_p=0.95,
            )
            return response.choices[0].message.content.strip()
        except APIConnectionError as e:
            if attempt == max_retries:
                raise
            wait = retry_delay * attempt
            print(f"  ⚠️  Connection error (attempt {attempt}/{max_retries}). "
                  f"Retrying in {wait:.0f}s… ({e})")
            time.sleep(wait)

# Pretty-print helper
def pretty(label: str, text: str):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(textwrap.fill(text, width=80) if '\n' not in text else text)

# Quick connectivity check
try:
    ping = call_gemma("Reply with only the word: READY")
    print(f"✅ Gemma 4 server is live. Model replied: '{ping}'")
except Exception as e:
    print(f"❌ Could not reach Ollama server: {e}")
    print("   Fix: brew services start ollama && ollama pull gemma4:e2b")

✅ Gemma 4 server is live. Model replied: 'READY'


---
## 📌 Experiment 1 — Reflexion Algorithm

### 🧩 Concept
The **Reflexion Algorithm** (Shinn et al.) adds a *self-critique loop* around the normal agent pipeline:

```
┌─────────────────────────────────────────────┐
│  1. Attempt  →  2. Reflect  →  3. Retry     │
│       ↑__________________________|           │
└─────────────────────────────────────────────┘
```

**Key idea:** Instead of accepting the first answer, the agent *analyzes its own output*, identifies flaws, and re-runs with that reflection as extra context.

### 🎯 What you will build
A Reflexion loop that:
1. Takes any task + a quality criterion
2. Gets an initial answer from Gemma 4
3. Has Gemma 4 reflect on whether the criterion is met
4. If not satisfied → retries with the reflection, up to `max_rounds`

In [4]:
# ─────────────────────────────────────────────────────────────
# Reflexion Algorithm — Core Implementation
# ─────────────────────────────────────────────────────────────

def reflexion_agent(
    task: str,
    criterion: str,
    max_rounds: int = 3,
    verbose: bool = True
) -> dict:
    """
    Run the Reflexion loop for a given task.

    Args:
        task      : The task prompt given to the agent.
        criterion : The quality bar the answer must meet.
        max_rounds: Maximum number of reflect-and-retry rounds.
        verbose   : Print round-by-round progress.

    Returns:
        A dict with keys: rounds, final_answer, history
    """
    history = []
    reflection_context = ""

    for round_num in range(1, max_rounds + 1):
        if verbose:
            print(f"\n🔄  Round {round_num}/{max_rounds}")

        # ── Step 1: Attempt ──────────────────────────────────
        attempt_prompt = task
        if reflection_context:
            attempt_prompt = (
                f"{task}\n\n"
                f"[Previous reflection]: {reflection_context}\n"
                f"Please improve your answer based on the reflection above."
            )

        answer = call_gemma(
            attempt_prompt,
            system="You are a helpful, precise assistant."
        )

        if verbose:
            pretty(f"📝 Answer (Round {round_num})", answer)

        # ── Step 2: Reflect ──────────────────────────────────
        reflect_prompt = (
            f"Task: {task}\n"
            f"Criterion for success: {criterion}\n"
            f"My answer: {answer}\n\n"
            f"Does my answer fully satisfy the criterion? "
            f"Reply in this exact JSON format:\n"
            f'{{"satisfied": true/false, "reflection": "<1-2 sentence critique>", "score": <1-10>}}'
        )

        raw_reflection = call_gemma(
            reflect_prompt,
            system="You are a strict quality-control agent. Be concise and honest."
        )

        # Parse JSON safely
        try:
            # Strip markdown code fences if present
            clean = raw_reflection.strip().removeprefix("```json").removesuffix("```").strip()
            reflection_data = json.loads(clean)
        except json.JSONDecodeError:
            reflection_data = {"satisfied": False, "reflection": raw_reflection, "score": 0}

        reflection_context = reflection_data.get("reflection", "")
        satisfied = reflection_data.get("satisfied", False)
        score = reflection_data.get("score", 0)

        if verbose:
            pretty(f"🪞 Reflection (Round {round_num})",
                   f"Score: {score}/10 | Satisfied: {satisfied}\n{reflection_context}")

        history.append({
            "round": round_num,
            "answer": answer,
            "reflection": reflection_context,
            "score": score,
            "satisfied": satisfied
        })

        if satisfied:
            if verbose:
                print(f"\n✅ Criterion met at round {round_num}. Stopping early.")
            break

    return {
        "rounds_used": round_num,
        "final_answer": answer,
        "history": history
    }

print("✅ Reflexion agent function defined.")

✅ Reflexion agent function defined.


In [5]:
# ─────────────────────────────────────────────────────────────
# Demo 1A — Summarization with strict bullet constraint
# ─────────────────────────────────────────────────────────────

task_1a = """
Summarize the following paragraph in EXACTLY 3 bullet points.
Each bullet must be under 15 words.

Paragraph:
The James Webb Space Telescope (JWST), launched in December 2021, is the
most powerful space observatory ever built. It observes the universe in
infrared light, allowing it to peer through cosmic dust clouds and see
the earliest galaxies formed just after the Big Bang. Its primary mirror
spans 6.5 meters and is made of 18 hexagonal gold-coated beryllium
segments. Scientists expect the JWST to transform our understanding of
planet formation, star birth, and the expansion of the universe.
"""

criterion_1a = (
    "The answer must contain EXACTLY 3 bullet points "
    "and each bullet must be under 15 words."
)

result_1a = reflexion_agent(task_1a, criterion_1a, max_rounds=3)


🔄  Round 1/3

  📝 Answer (Round 1)
*   JWST is the most powerful space observatory ever built.
*   It observes the universe using infrared light to see early galaxies.
*   It will transform understanding of star birth and cosmic expansion.

  🪞 Reflection (Round 1)
Score: 10/10 | Satisfied: True
The answer successfully summarized the main points of the paragraph in exactly three concise bullet points, adhering to the word limit.

✅ Criterion met at round 1. Stopping early.


In [6]:
# ─────────────────────────────────────────────────────────────
# Demo 1B — Code generation: must contain error handling
# ─────────────────────────────────────────────────────────────

task_1b = """
Write a Python function called `safe_divide(a, b)` that divides a by b.
"""

criterion_1b = (
    "The function MUST handle ZeroDivisionError gracefully and "
    "return None (not raise) when b is 0. Also include a docstring."
)

result_1b = reflexion_agent(task_1b, criterion_1b, max_rounds=3)


🔄  Round 1/3

  📝 Answer (Round 1)
This is a common scenario where you need to handle potential errors, specifically **division by zero**. A "safe" function should check for this condition before attempting the calculation.

Here are a few ways to implement `safe_divide`, depending on how you want the function to handle the error:

### Option 1: Raising a `ValueError` (Recommended)

This is the most standard and robust way in Python. If division by zero is attempted, the function raises an explicit error, allowing the calling program to handle the failure.

```python
def safe_divide(a, b):
    """
    Divides 'a' by 'b', safely checking for division by zero.

    Args:
        a (float/int): The numerator.
        b (float/int): The denominator.

    Returns:
        float/int: The result of the division (a / b).

    Raises:
        ValueError: If the denominator 'b' is zero.
    """
    if b == 0:
        raise ValueError("Error: Cannot divide by zero.")
    
    return a / b

# ---

In [7]:
# ─────────────────────────────────────────────────────────────
# Inspect detailed history of any run
# ─────────────────────────────────────────────────────────────

print("\n📊 Reflexion History — Demo 1A\n")
for entry in result_1a["history"]:
    print(f"  Round {entry['round']} | Score {entry['score']}/10 | Satisfied: {entry['satisfied']}")
    print(f"    Reflection: {entry['reflection']}")


📊 Reflexion History — Demo 1A

  Round 1 | Score 10/10 | Satisfied: True
    Reflection: The answer successfully summarized the main points of the paragraph in exactly three concise bullet points, adhering to the word limit.


### 🤔 Comprehension Check — Reflexion

1. What prevents the Reflexion loop from running forever?
2. What happens if the model *always* marks itself as satisfied immediately?
3. Try changing `max_rounds=1` — what changes in quality?
4. *Extension:* Add a `temperature` parameter that increases slightly each round — why might that help?

---

## 📌 Experiment 2 — Auto-Verbalization Grading

### 🧩 Concept
**Auto-Verbalization Grading** makes the model *speak aloud* about the quality of its own answer — like a student explaining to a teacher why they believe their answer is correct (or not).

Unlike Reflexion which retries, Auto-Verbalization Grading is primarily a **confidence & quality signal** — useful when:
- You have no ground truth to compare against
- You want to detect hallucinations before serving the answer
- You're building evaluation pipelines

```
User Question
     ↓
  [Agent answers]
     ↓
  [Agent grades itself with verbal rationale]
     ↓
  Grade + Confidence + Explanation
```

### 🎯 What you will build
A two-step pipeline: solve → self-grade with structured verbalization.

In [8]:
# ─────────────────────────────────────────────────────────────
# Auto-Verbalization Grading — Core Implementation
# ─────────────────────────────────────────────────────────────

def auto_verbalization_grade(
    question: str,
    domain_hint: str = "",
    verbose: bool = True
) -> dict:
    """
    Two-step pipeline:
      Step 1 — Agent answers the question.
      Step 2 — Agent verbalizes a self-grade with rationale.

    Args:
        question    : The question to answer.
        domain_hint : Optional domain context (e.g. 'geography', 'math').
        verbose     : Print step-by-step output.

    Returns:
        dict with: question, answer, grade, confidence, rationale, flag
    """

    # ── Step 1: Answer ────────────────────────────────────────
    answer_prompt = question
    if domain_hint:
        answer_prompt = f"[Domain: {domain_hint}]\n{question}"

    answer = call_gemma(
        answer_prompt,
        system="You are a knowledgeable assistant. Answer directly and concisely."
    )

    if verbose:
        pretty("📝 Agent Answer", answer)

    # ── Step 2: Self-Grade ────────────────────────────────────
    grading_prompt = f"""
You have answered the following question:

Question: {question}
Your Answer: {answer}

Now grade yourself honestly. Think step-by-step:
1. Is the answer factually correct to the best of your knowledge?
2. Is it complete and specific?
3. Could it be a hallucination?

Respond in this EXACT JSON format:
{{
  "grade": <integer 1-10>,
  "confidence": "high" | "medium" | "low",
  "rationale": "<2-3 sentence verbal explanation of your grade>",
  "potential_error": "<describe any specific mistake or uncertainty>",
  "flag_for_review": true | false
}}
"""

    raw_grade = call_gemma(
        grading_prompt,
        system="You are a rigorous self-evaluator. Be honest — it is better to admit uncertainty than to pretend confidence."
    )

    try:
        clean = raw_grade.strip().removeprefix("```json").removesuffix("```").strip()
        grade_data = json.loads(clean)
    except json.JSONDecodeError:
        grade_data = {
            "grade": 0, "confidence": "unknown",
            "rationale": raw_grade,
            "potential_error": "Could not parse JSON",
            "flag_for_review": True
        }

    if verbose:
        pretty("🪞 Self-Grade",
               f"Grade: {grade_data.get('grade')}/10 | "
               f"Confidence: {grade_data.get('confidence')} | "
               f"Flag: {grade_data.get('flag_for_review')}\n"
               f"Rationale: {grade_data.get('rationale')}\n"
               f"Potential Error: {grade_data.get('potential_error')}")

    return {
        "question": question,
        "answer": answer,
        **grade_data
    }

print("✅ Auto-Verbalization Grading function defined.")

✅ Auto-Verbalization Grading function defined.


In [9]:
# ─────────────────────────────────────────────────────────────
# Demo 2A — Translate a word (from the lecture example)
# ─────────────────────────────────────────────────────────────

result_2a = auto_verbalization_grade(
    question="What does the Spanish word 'libro' mean in English?",
    domain_hint="Spanish language"
)


  📝 Agent Answer
Book

  🪞 Self-Grade
Grade: 10/10 | Confidence: high | Flag: False
Rationale: The answer is factually correct; 'libro' is the standard Spanish word for 'book'. The response is complete and directly answers the question without ambiguity.
Potential Error: No potential error; the translation is standard and universally accepted.


In [10]:
# ─────────────────────────────────────────────────────────────
# Demo 2B — Tricky factual question (capital city)
# ─────────────────────────────────────────────────────────────

result_2b = auto_verbalization_grade(
    question="What is the capital city of Australia?",
    domain_hint="geography"
)


  📝 Agent Answer
Canberra is the capital city of Australia.

  🪞 Self-Grade
Grade: 10/10 | Confidence: high | Flag: False
Rationale: The answer is factually correct and directly addresses the user's question. Canberra is, in fact, the capital city of Australia.
Potential Error: None. The information provided is accurate.


In [11]:
# ─────────────────────────────────────────────────────────────
# Demo 2C — Deliberate hallucination trap (obscure/false premise)
# ─────────────────────────────────────────────────────────────

result_2c = auto_verbalization_grade(
    question="Who won the Nobel Peace Prize in 2087?",
    domain_hint="history"
)


  📝 Agent Answer
I do not have information about who will win the Nobel Peace Prize in 2087, as
that event has not yet occurred.

  🪞 Self-Grade
Grade: 10/10 | Confidence: high | Flag: False
Rationale: The answer is factually correct because the event in 2087 has not yet occurred, making it impossible to know the outcome. I correctly state the limitation of my knowledge regarding future, unverified events.
Potential Error: None. The answer accurately reflects the impossibility of predicting a future award winner.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Batch evaluation — run multiple questions and show a summary table
# ─────────────────────────────────────────────────────────────

questions_batch = [
    ("What is the speed of light in m/s?",         "physics"),
    ("Who wrote the play Hamlet?",                  "literature"),
    ("What is the chemical symbol for gold?",       "chemistry"),
    ("What year did World War I end?",              "history"),
    ("Who invented the programming language Python?", "computer science"),
]

batch_results = []
for q, d in questions_batch:
    r = auto_verbalization_grade(q, domain_hint=d, verbose=False)
    batch_results.append(r)

print("\n📊 Batch Evaluation Summary")
print(f"{'Question':<50} {'Grade':>6} {'Conf':>8} {'Flag':>6}")
print("-" * 75)
for r in batch_results:
    print(f"{r['question'][:48]:<50} "
          f"{r.get('grade', '?'):>6} "
          f"{r.get('confidence', '?'):>8} "
          f"{str(r.get('flag_for_review', '?')):>6}")

### 🤔 Comprehension Check — Auto-Verbalization Grading

1. Why is `flag_for_review` more useful than just the numeric grade?
2. What did the model do when asked about a future Nobel Prize winner (2087)?
3. How would you integrate this into a real chatbot — when would you show the grade to the user vs. hide it?
4. *Extension:* Add a second-pass that only retries answers flagged with `flag_for_review=True`.

---

## 📌 Experiment 3 — AutoGen Evaluator Sub-Agent

### 🧩 Concept
The **AutoGen Evaluator Sub-Agent** is an independent judge agent that critiques the output of a *main agent*. This is analogous to peer review in science:

```
User Prompt
     ↓
  [Main Agent]  →  Draft Answer
                        ↓
              [Evaluator Sub-Agent]
                        ↓
         Score | Rationale | Suggestions
                        ↓
   (Optional) [Main Agent revises with feedback]
```

**Why a separate agent?**
- Separation of concerns — the generator doesn't self-censor
- Can use a *stronger* model or *stricter* system prompt for evaluation
- Enables ensemble pipelines and feedback loops

### 🎯 What you will build
1. A configurable Main Agent
2. A configurable Evaluator Sub-Agent with structured critique
3. An optional revision loop where the main agent incorporates feedback

In [ ]:
# ─────────────────────────────────────────────────────────────
# Evaluator Sub-Agent — Core Implementation
# ─────────────────────────────────────────────────────────────

def main_agent(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """The primary task-solving agent."""
    return call_gemma(prompt, system=system)


def evaluator_sub_agent(
    original_prompt: str,
    agent_response: str,
    evaluation_rubric: str = "",
    verbose: bool = True
) -> dict:
    """
    Evaluates the main agent's response against the original prompt.

    Args:
        original_prompt  : The original task/question.
        agent_response   : The main agent's answer to evaluate.
        evaluation_rubric: Optional rubric / evaluation criteria.
        verbose          : Print evaluation output.

    Returns:
        dict with: score, verdict, critique, suggestions, corrected_fact
    """
    rubric_text = f"\nEvaluation rubric: {evaluation_rubric}" if evaluation_rubric else ""

    eval_prompt = f"""
You are a strict peer-review evaluator. Your job is to critique the following agent response.

Original Prompt: {original_prompt}
Agent Response: {agent_response}{rubric_text}

Evaluate the response on these dimensions:
- Factual Accuracy
- Completeness
- Clarity
- Relevance to the prompt

Reply in this EXACT JSON format:
{{
  "score": <integer 1-10>,
  "verdict": "pass" | "fail" | "partial",
  "critique": "<2-3 sentence critique covering all dimensions>",
  "specific_errors": ["<error 1>", "<error 2>"],
  "suggestions": ["<improvement 1>", "<improvement 2>"],
  "corrected_fact": "<corrected answer if the agent was factually wrong, else null>"
}}
"""

    raw_eval = call_gemma(
        eval_prompt,
        system="You are an impartial, expert evaluator. Be specific and rigorous."
    )

    try:
        clean = raw_eval.strip().removeprefix("```json").removesuffix("```").strip()
        eval_data = json.loads(clean)
    except json.JSONDecodeError:
        eval_data = {
            "score": 0, "verdict": "unknown",
            "critique": raw_eval, "specific_errors": [],
            "suggestions": [], "corrected_fact": None
        }

    if verbose:
        pretty("⚖️  Evaluator Sub-Agent Critique",
               f"Score: {eval_data.get('score')}/10 | Verdict: {(eval_data.get('verdict') or 'unknown').upper()}\n"
               f"Critique: {eval_data.get('critique')}\n"
               f"Errors: {eval_data.get('specific_errors')}\n"
               f"Suggestions: {eval_data.get('suggestions')}\n"
               f"Corrected Fact: {eval_data.get('corrected_fact')}")

    return eval_data


def autogen_pipeline(
    user_prompt: str,
    rubric: str = "",
    revise_if_fail: bool = True,
    verbose: bool = True
) -> dict:
    """
    Full AutoGen-style pipeline:
      Main Agent → Evaluator → (optional revision)

    Args:
        user_prompt    : The task prompt.
        rubric         : Optional evaluation rubric.
        revise_if_fail : If the evaluator gives a 'fail', ask the main agent to revise.
        verbose        : Print all steps.

    Returns:
        dict with full pipeline results.
    """
    print("\n" + "─"*60)
    print(f"📨 User Prompt: {user_prompt}")
    print("─"*60)

    # Step 1: Main Agent answers
    draft = main_agent(user_prompt)
    if verbose:
        pretty("🤖 Main Agent Draft", draft)

    # Step 2: Evaluator critiques
    evaluation = evaluator_sub_agent(
        original_prompt=user_prompt,
        agent_response=draft,
        evaluation_rubric=rubric,
        verbose=verbose
    )

    revised = None

    # Step 3 (optional): Revise if failed
    if revise_if_fail and evaluation.get("verdict") == "fail":
        if verbose:
            print("\n♻️  Verdict is FAIL — asking main agent to revise with feedback...")

        revision_prompt = (
            f"Original task: {user_prompt}\n\n"
            f"Your previous answer: {draft}\n\n"
            f"Evaluator feedback:\n"
            f"  Critique: {evaluation.get('critique')}\n"
            f"  Errors: {evaluation.get('specific_errors')}\n"
            f"  Suggestions: {evaluation.get('suggestions')}\n"
            f"  Corrected fact: {evaluation.get('corrected_fact')}\n\n"
            f"Please provide an improved answer addressing all the feedback."
        )
        revised = main_agent(revision_prompt)
        if verbose:
            pretty("✏️  Revised Answer", revised)

    return {
        "prompt": user_prompt,
        "draft": draft,
        "evaluation": evaluation,
        "revised": revised
    }

print("✅ AutoGen Evaluator pipeline defined.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo 3A — Lecture example: capital of Brazil
# ─────────────────────────────────────────────────────────────

result_3a = autogen_pipeline(
    user_prompt="What is the capital city of Brazil? Explain why it is significant.",
    rubric="Answer must name the correct capital and provide at least one historical or political reason for its significance.",
    revise_if_fail=True
)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo 3B — Code review: evaluator acts as a senior engineer
# ─────────────────────────────────────────────────────────────

result_3b = autogen_pipeline(
    user_prompt=(
        "Write a Python function that reads a CSV file and returns "
        "the average of a specified numeric column."
    ),
    rubric=(
        "Code must: (1) handle FileNotFoundError, "
        "(2) handle missing/non-numeric values, "
        "(3) include a docstring, "
        "(4) be PEP-8 compliant."
    ),
    revise_if_fail=True
)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo 3C — Creative writing evaluation
# ─────────────────────────────────────────────────────────────

result_3c = autogen_pipeline(
    user_prompt="Write a 3-sentence product description for a smart water bottle.",
    rubric=(
        "Must: mention at least one smart feature (app, sensor, etc.), "
        "include a benefit for the user's health, "
        "and be written in an engaging, marketing tone."
    ),
    revise_if_fail=True
)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Compare draft vs revised for Demo 3B
# ─────────────────────────────────────────────────────────────

print("\n📄 Draft vs Revised Comparison — Demo 3B")
print("\n--- DRAFT ---")
print(result_3b["draft"])

if result_3b["revised"]:
    print("\n--- REVISED ---")
    print(result_3b["revised"])
else:
    print("\n(No revision needed — draft passed evaluation)")

### 🔬 Advanced: Multi-Agent Ensemble with Majority Voting

Run 3 evaluators independently and take the **majority verdict** — reduces individual model bias.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Ensemble Evaluator — 3 independent critique agents vote
# ─────────────────────────────────────────────────────────────

def ensemble_evaluator(
    prompt: str,
    response: str,
    rubric: str = "",
    n_evaluators: int = 3
) -> dict:
    """
    Run `n_evaluators` independent evaluations and aggregate results
    using majority voting on verdict and average score.
    """
    evaluations = []
    for i in range(n_evaluators):
        print(f"  ↳ Evaluator {i+1}/{n_evaluators} running...")
        ev = evaluator_sub_agent(prompt, response, rubric, verbose=False)
        evaluations.append(ev)

    # Aggregate
    scores = [e.get("score", 0) for e in evaluations]
    verdicts = [e.get("verdict", "unknown") for e in evaluations]
    avg_score = sum(scores) / len(scores)
    majority_verdict = max(set(verdicts), key=verdicts.count)

    print(f"\n📊 Ensemble Result:")
    print(f"   Individual scores  : {scores}")
    print(f"   Average score      : {avg_score:.1f}/10")
    print(f"   Individual verdicts: {verdicts}")
    print(f"   Majority verdict   : {majority_verdict.upper()}")

    return {
        "scores": scores,
        "avg_score": avg_score,
        "verdicts": verdicts,
        "majority_verdict": majority_verdict,
        "evaluations": evaluations
    }


# Test ensemble on a debatable answer
test_prompt   = "Is Python a good language for building production-grade web APIs?"
test_response = main_agent(test_prompt)
pretty("🤖 Main Agent", test_response)

print("\n🗳️  Running 3-evaluator ensemble...")
ensemble_result = ensemble_evaluator(
    prompt=test_prompt,
    response=test_response,
    rubric="Answer should be balanced, mention real-world frameworks, and address trade-offs."
)

### 🤔 Comprehension Check — AutoGen Evaluator

1. What is the key difference between the Reflexion Algorithm and the AutoGen Evaluator pattern?
2. Why might you use a *stronger* model (e.g., a larger Gemma variant) for the evaluator than for the main agent?
3. When would an ensemble evaluator be worth the extra compute cost?
4. *Extension:* Modify `autogen_pipeline` to allow up to 2 revisions (not just 1) if the score stays below 7.

---

## 📊 Summary — All Three Techniques Side by Side

In [ ]:
# ─────────────────────────────────────────────────────────────
# Head-to-head comparison on the same task
# ─────────────────────────────────────────────────────────────

shared_task = "Explain the concept of recursion to a 10-year-old in 3 sentences."
shared_criterion = "Must use a real-world analogy, be under 60 words total, and use simple vocabulary."

print("\n" + "="*70)
print("COMPARATIVE RUN — Same Task Through All 3 Techniques")
print("="*70)
print(f"Task: {shared_task}")
print(f"Criterion: {shared_criterion}")

# --- Technique 1: Reflexion ---
print("\n" + "─"*40)
print("TECHNIQUE 1: Reflexion Algorithm")
print("─"*40)
r1 = reflexion_agent(shared_task, shared_criterion, max_rounds=2)

# --- Technique 2: Auto-Verbalization ---
print("\n" + "─"*40)
print("TECHNIQUE 2: Auto-Verbalization Grading")
print("─"*40)
r2 = auto_verbalization_grade(shared_task)

# --- Technique 3: AutoGen Evaluator ---
print("\n" + "─"*40)
print("TECHNIQUE 3: AutoGen Evaluator Sub-Agent")
print("─"*40)
r3 = autogen_pipeline(shared_task, rubric=shared_criterion, revise_if_fail=True)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Summary table
# ─────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("SUMMARY TABLE")
print("="*70)
print(f"{'Technique':<35} {'Purpose':<30} {'Key Output'}")
print("-"*85)

rows = [
    ("Reflexion Algorithm",
     "Self-corrective retries",
     "Improved answer after N rounds"),
    ("Auto-Verbalization Grading",
     "Self-assessed confidence",
     "Grade + rationale + flag"),
    ("AutoGen Evaluator Sub-Agent",
     "Structured peer review",
     "Score + verdict + suggestions + optional revision"),
]

for technique, purpose, output in rows:
    print(f"{technique:<35} {purpose:<30} {output}")

print("\n✅ Tutorial complete — all three techniques demonstrated.")

---
## 🚀 Next Steps & Extensions

| Challenge | Description |
|-----------|-------------|
| **Chain all 3** | Build a pipeline: Reflexion → Auto-Grade → Evaluator |
| **Add memory** | Store reflections in a vector DB and inject past lessons |
| **Fine-tune rubrics** | Use domain-specific rubrics for medical, legal, or financial QA |
| **Async execution** | Run evaluator calls in parallel with `asyncio` for speed |
| **Cost tracking** | Log token counts per technique and compare efficiency |
| **Human-in-the-loop** | Pause the loop when `flag_for_review=True` and ask a human |

---
**References**  
- Shinn et al. (2023) *Reflexion: Language Agents with Verbal Reinforcement Learning* — [arXiv:2303.11366](https://arxiv.org/abs/2303.11366)  
- Gemma 4 E2B on Ollama — `ollama pull gemma4:e2b`  
- Ollama OpenAI-compatible API — `http://localhost:11434/v1`  
- AutoGen Framework — https://microsoft.github.io/autogen/  

*Boston Institute of Analytics — Self-Reflection & Critique Lecture*